<a href="https://colab.research.google.com/github/mlbyhoang/marketing-attribution/blob/main/notebooks/03_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 – Logistic Regression Baseline

This notebook develops and evaluates a Logistic Regression baseline for journey-level conversion prediction and compares different imbalance-handling strategies.

In [ ]:
# Load libraries
import os
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
from sklearn.preprocessing import (
    MultiLabelBinarizer,
    StandardScaler

)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)
from scipy.stats import loguniform
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import ADASYN, SMOTE
from sklearn.metrics import (
    auc,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    brier_score_loss,
    precision_recall_curve,
    precision_score,
    recall_score, roc_auc_score, roc_curve,
)

warnings.filterwarnings('ignore')

In [ ]:
# Connect to Google Drive
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
# Load processed data path from user data / secrets
FILTERED_DATA_PATH = userdata.get("MTA_FILTERED_PATH")

In [ ]:
# Load processed dataset and inspect structure
df_filtered = pd.read_parquet(FILTERED_DATA_PATH)

print("Shape:", df_filtered.shape)
print("Columns:", df_filtered.columns.tolist())
print(f"Conversion Rate: {df_filtered['order'].mean():.2%}")

In [ ]:
# Display first few rows
df_filtered.head()

In [ ]:
# Define features
time_feature_cols = [
    # Interaction time features
    "interaction_hour_sin",
    "interaction_hour_cos",
    "interaction_weekday_sin",
    "interaction_weekday_cos",

    # Journey features
    "tp_time_delta_min",
    "interaction_number",

    # Interaction type
    "is_click"
]

# Derived feature groups
sincos_cols   = [
    k for k in time_feature_cols
    if '_sin' in k or '_cos' in k
    ]

journey_cols  = [
    "tp_time_delta_min",
    "interaction_number"
    ]

binary_cols = [
    "is_click"
    ]

In [ ]:
# Aggregate touchpoint-level features to journey-level features
def aggregate_seq_times(seq_times):

    if len(seq_times) == 0:
        return {}

    keys = list(seq_times[0].keys())

    arr = np.array([
        [d[key] for key in keys]
        for d in seq_times
    ])

    result = {}

    for i, key in enumerate(keys):

        if key in sincos_cols:
            result[f"{key}_mean"] = arr[:, i].mean()
            result[f"{key}_last"] = arr[-1, i]

        elif key in journey_cols:
            result[f"{key}_mean"] = arr[:, i].mean()
            result[f"{key}_max"] = arr[:, i].max()
            result[f"{key}_std"] = arr[:, i].std()

        elif key in binary_cols:
            result[f"{key}_mean"] = arr[:, i].mean()
            result[f"{key}_sum"] = arr[:, i].sum()

    return result

In [ ]:
# Aggregate features for all journeys
df_time_features = pd.DataFrame(
    df_filtered["seq_times"]
    .apply(aggregate_seq_times)
    .tolist()
)

print("Shape:", df_time_features.shape)
print(f"Missing values: {df_time_features.isna().sum().sum()}")

In [ ]:
# Combine filtered journeys with aggregated features
df_model = pd.concat([
    df_filtered.reset_index(drop=True),
    df_time_features.reset_index(drop=True)
], axis=1)

print("Shape:", df_model.shape)
print(f"Conversion Rate: {df_model['order'].mean():.2%}")
print(f"Missing values: {df_model.isna().sum().sum()}")
print(f"Columns: {df_model.columns.tolist()}")

In [ ]:
# Display DataFrame summary and data types
df_model.info()

In [ ]:
# Display summary statistics
df_model.describe()

In [ ]:
# One-hot encode channel presence in each journey
mlb = MultiLabelBinarizer()

ohe = mlb.fit_transform(df_model["sequence"])

ohe_df = pd.DataFrame(
    ohe,
    columns=mlb.classes_,
    index=df_model.index
)

print(f"Anzahl Touchpoint-Typen: {len(mlb.classes_)}")
print(f"Shape OHE:               {ohe_df.shape}")
print(f"Touchpoint-Typen:        {mlb.classes_}")

In [ ]:
# Combine model features with one-hot encoded channels
df_model_ohe = pd.concat([
    df_model.reset_index(drop=True),
    ohe_df.reset_index(drop=True)
], axis=1)

print("Shape:", df_model_ohe.shape)
print(f"Missing values: {df_model_ohe.isna().sum().sum()}")
print(f"Conversion Rate: {df_model_ohe['order'].mean():.2%}")

In [ ]:
# Define target
y = df_model_ohe["order"].copy()

# Remove non-model features and target
X = df_model_ohe.drop(
    columns=[
        "ga_cid",
        "sequence",
        "seq_times",
        "order"
    ]
).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print(f"Conversion Rate: {y.mean():.2%}")
print(f"Missing values in X: {X.isna().sum().sum()}")
print(f"Duplicate columns: {X.columns.duplicated().sum()}")

In [ ]:
# Train / validation / test split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y_trainval
)

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_val:   {X_val.shape}")
print(f"Shape X_test:  {X_test.shape}")

print(f"\nCR Train: {y_train.mean():.2%}")
print(f"CR Val:   {y_val.mean():.2%}")
print(f"CR Test:  {y_test.mean():.2%}")

In [ ]:
# Cross-validation setup
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
# Model 1: Logistic Regression with class weighting
param_dist_cw = {
    "lr__C": loguniform(0.001, 100),
    "lr__penalty": ["l1", "l2"],
    "lr__solver": ["liblinear"],
    "lr__max_iter": [500, 1000],
    "lr__class_weight": [
        {0: 1.0, 1: 5.0},
        {0: 1.0, 1: 7.0},
        {0: 1.0, 1: 10.0},
        {0: 1.0, 1: 13.0},
        "balanced"
    ]
}

pipeline_cw =  ImbPipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(random_state=42))
])

random_cw = RandomizedSearchCV(
    estimator=pipeline_cw,
    param_distributions=param_dist_cw,
    n_iter=25,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_cw.fit(X_train, y_train)

best_lr_cw = random_cw.best_estimator_
best_cv_pr_auc_cw = random_cw.best_score_

print(f"Best Params: {random_cw.best_params_}")
print(f"Best PR-AUC (CV): {best_cv_pr_auc_cw:.4f}")

In [ ]:
# Model 2: Logistic Regression with ADASYN
param_dist_adasyn = {
    "lr__C": loguniform(0.001, 100),
    "lr__penalty": ["l1", "l2"],
    "lr__solver": ["liblinear"],
    "lr__max_iter": [500, 1000],
    "adasyn__sampling_strategy": [0.3, 0.5, 1.0]
}

pipeline_adasyn = ImbPipeline([
    ("scaler", StandardScaler()),
    ("adasyn", ADASYN(random_state=42)),
    ("lr", LogisticRegression(random_state=42))
])

random_adasyn = RandomizedSearchCV(
    estimator=pipeline_adasyn,
    param_distributions=param_dist_adasyn,
    n_iter=25,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_adasyn.fit(X_train, y_train)

best_lr_adasyn = random_adasyn.best_estimator_
best_cv_pr_auc_adasyn = random_adasyn.best_score_

print(f"Best Params (ADASYN): {random_adasyn.best_params_}")
print(f"Best PR-AUC (CV): {best_cv_pr_auc_adasyn:.4f}")

In [ ]:
# Model 3: Logistic Regression with class weighting + ADASYN
param_dist_both = {
    "lr__C": loguniform(0.001, 100),
    "lr__penalty": ["l1", "l2"],
    "lr__solver": ["liblinear"],
    "lr__max_iter": [500, 1000],
    "lr__class_weight": [
        {0: 1.0, 1: 5.0},
        {0: 1.0, 1: 7.0},
        {0: 1.0, 1: 10.0},
        "balanced"
    ],
    "adasyn__sampling_strategy": [0.3, 0.5, 1.0]
}

pipeline_both = ImbPipeline([
    ("scaler", StandardScaler()),
    ("adasyn", ADASYN(random_state=42)),
    ("lr", LogisticRegression(random_state=42))
])

random_both = RandomizedSearchCV(
    estimator=pipeline_both,
    param_distributions=param_dist_both,
    n_iter=25,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_both.fit(X_train, y_train)

best_lr_both = random_both.best_estimator_
best_cv_pr_auc_both = random_both.best_score_

print(f"Best Params (Both): {random_both.best_params_}")
print(f"Best PR-AUC (CV): {best_cv_pr_auc_both:.4f}")

In [ ]:
# Model 4: Baseline Logistic Regression without imbalance handling
param_dist_base = {
    "lr__C": loguniform(0.001, 100),
    "lr__penalty": ["l1", "l2"],
    "lr__solver": ["liblinear"],
    "lr__max_iter": [500, 1000]
}

pipeline_base = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(random_state=42))
])

random_base = RandomizedSearchCV(
    estimator=pipeline_base,
    param_distributions=param_dist_base,
    n_iter=25,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_base.fit(X_train, y_train)

best_lr_base = random_base.best_estimator_
best_cv_pr_auc_base = random_base.best_score_

print(f"Best Params (Baseline): {random_base.best_params_}")
print(f"Best PR-AUC (CV): {best_cv_pr_auc_base:.4f}")

In [ ]:
# Find threshold that maximizes F1-score
def optimal_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        y_prob
    )

    precision = precision[:-1]
    recall = recall[:-1]

    f1_scores = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    best_idx = np.argmax(f1_scores)

    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    y_pred_opt = (y_prob >= best_threshold).astype(int)

    precision_opt = precision_score(y_true, y_pred_opt)
    recall_opt = recall_score(y_true, y_pred_opt)

    report = classification_report(
        y_true,
        y_pred_opt
    )

    return (
        best_threshold,
        best_f1,
        precision_opt,
        recall_opt,
        report
    )

In [ ]:
models = {
    "Baseline LR": best_lr_base,
    "LR + Class Weights": best_lr_cw,
    "LR + ADASYN": best_lr_adasyn,
    "LR + Class Weights + ADASYN": best_lr_both
}

val_results = []

for name, model in models.items():

    y_val_prob = model.predict_proba(X_val)[:, 1]

    pr_auc = average_precision_score(y_val, y_val_prob)
    roc_auc = roc_auc_score(y_val, y_val_prob)

    threshold, f1, precision, recall, _ = optimal_threshold(
        y_val,
        y_val_prob
    )

    val_results.append({
        "Model": name,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Threshold": threshold,
        "F1": f1,
        "Precision": precision,
        "Recall": recall
    })

val_results = pd.DataFrame(val_results)

val_results.sort_values(
    "PR_AUC",
    ascending=False
).reset_index(drop=True)

In [ ]:
# Select final model based on CV and validation performance
final_model = best_lr_base

# Fix classification threshold based on validation data
y_prob_val = final_model.predict_proba(X_val)[:, 1]

final_threshold, _, _, _, _ = optimal_threshold(
    y_val,
    y_prob_val
)

print(f"Selected model: Baseline Logistic Regression")
print(f"Final threshold: {final_threshold:.4f}")

In [ ]:
# Predict probabilities on test set
y_prob_test = final_model.predict_proba(X_test)[:, 1]

# Apply threshold selected on validation data
y_pred_test = (
    y_prob_test >= final_threshold
).astype(int)

# Threshold-independent metrics
test_pr_auc = average_precision_score(
    y_test,
    y_prob_test
)

test_roc_auc = roc_auc_score(
    y_test,
    y_prob_test
)

print("===== Final Test Performance =====")
print(f"PR-AUC:    {test_pr_auc:.4f}")
print(f"ROC-AUC:   {test_roc_auc:.4f}")
print(f"Threshold: {final_threshold:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_test
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_test
    )
)

In [ ]:
# Save final Logistic Regression results
save_path = userdata.get("MTA_RESULTS_PATH")

y_prob_lr_val = final_model.predict_proba(X_val)[:, 1]
y_prob_lr_test = final_model.predict_proba(X_test)[:, 1]

np.save(f"{save_path}/y_prob_lr_val.npy", y_prob_lr_val)
np.save(f"{save_path}/y_prob_lr_test.npy", y_prob_lr_test)
np.save(f"{save_path}/y_val.npy", y_val.to_numpy())
np.save(f"{save_path}/y_test.npy", y_test.to_numpy())

results_lr = {
    "model": "Baseline Logistic Regression",
    "threshold": float(final_threshold),
    "pr_auc": float(
        average_precision_score(y_test, y_prob_lr_test)
    ),
    "roc_auc": float(
        roc_auc_score(y_test, y_prob_lr_test)
    )
}

with open(f"{save_path}/results_lr.json", "w") as f:
    json.dump(results_lr, f, indent=2)

print("Final Logistic Regression results saved successfully.")
print(f"Model:     {results_lr['model']}")
print(f"Threshold: {results_lr['threshold']:.4f}")
print(f"PR-AUC:    {results_lr['pr_auc']:.4f}")
print(f"ROC-AUC:   {results_lr['roc_auc']:.4f}")